# GT-supervised auxiliary Deformable DETR

이 노트북은 실행 orchestration만 담당합니다. 데이터·모델·학습·평가 함수는 `gt_aux/`의 Python 모듈에 있습니다.

학습은 모델별로 독립된 셀에서 실행합니다. 각 셀은 하나의 모델만 학습하고 checkpoint와 CSV를 저장합니다.

## 실행 순서

1. 설정·데이터 준비 셀을 실행합니다.
2. architecture check 셀을 실행합니다.
3. baseline, shared_detach, shared_e2e 학습 셀을 필요한 순서로 각각 실행합니다.
4. 마지막 평가 셀에서 저장된 checkpoint들을 비교합니다.

학습 셀을 다시 실행하면 해당 모델만 새로 초기화되어 checkpoint를 덮어씁니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (Path('D:/gt-super') / 'data').exists():
    ROOT = Path('D:/gt-super')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from gt_aux.config import ExperimentConfig
from gt_aux.data import prepare_data
from gt_aux.eval import (
    evaluate_saved_experiment, load_histories, load_checkpoint,
    plot_gradient_history, plot_histories, summarize_histories,
    visualize_main_predictions,
)
from gt_aux.model import assert_hf_bbox_heads_are_tied, make_model
from gt_aux.train import release_model, train_one_experiment

CONFIG = ExperimentConfig(ROOT, run_mode='full')
print(CONFIG.as_dict())

d:\gt-super\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'root': 'D:\\gt-super', 'run_mode': 'full', 'train_images': 3000, 'val_images': 750, 'epochs': 7, 'batch_size': 2, 'device': 'cuda', 'experiments': ['baseline', 'shared_detach', 'shared_e2e'], 'seed': 42}


In [2]:
# 데이터는 한 번만 준비하고, 각 학습 셀은 이 bundle을 공유합니다.
BUNDLE = prepare_data(CONFIG)
print({
    'train_images': len(BUNDLE.train_records),
    'val_images': len(BUNDLE.val_records),
    'output_dir': str(CONFIG.output_dir),
})

VOC XML: 100%|██████████| 3750/3750 [00:02<00:00, 1650.10it/s]

Full split: train=3000 (9180 objects), val=750 (2530 objects)
Current run: train=3000, val=750
{'train_images': 3000, 'val_images': 750, 'output_dir': 'D:\\gt-super\\outputs\\detr_gt_auxiliary'}


In [3]:
# 모델 초기화와 shared bbox predictor 구조만 확인합니다. 학습은 하지 않습니다.
probe_model, probe_fingerprint = make_model(CONFIG, 'shared_e2e', CONFIG.seed)
assert_hf_bbox_heads_are_tied(probe_model.detector)
print({'model_fingerprint': probe_fingerprint, 'bbox_head_shared': True})
release_model(probe_model)

Loading weights: 100%|██████████| 545/545 [00:00<00:00, 16238.75it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

{'model_fingerprint': 'c16286f63961237b', 'bbox_head_shared': True}


## 1. Baseline

아래 셀만 실행하면 baseline 하나만 학습합니다.

In [ ]:
baseline_model, baseline_history, baseline_gradients = train_one_experiment(
    CONFIG, BUNDLE, experiment='baseline', seed=CONFIG.seed
)
release_model(baseline_model)
print('Saved:', CONFIG.checkpoint_path('baseline'))


===== baseline / seed=42 =====


Loading weights: 100%|██████████| 545/545 [00:00<00:00, 15874.83it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

[phase] initial main-only validation: 375 batches


[phase] initial validation complete: mAP=0.0001, AP@0.5=0.0002
[phase] training epoch 1/7: 1500 batches


[phase] validating epoch 1/7: 375 batches


{'epoch': 1, 'main_loss': 6.349, 'aux_loss': nan, 'aux_coverage': nan, 'collision_rate': nan, 'map': 0.037, 'map50': 0.0886, 'map75': 0.0254}
[phase] training epoch 2/7: 1500 batches


baseline e2:  55%|█████▌    | 827/1500 [20:40<28:05,  2.50s/it]  

## 2. Shared localization head + detached auxiliary path

이 셀은 `shared_detach`만 학습합니다.

In [ ]:
detach_model, detach_history, detach_gradients = train_one_experiment(
    CONFIG, BUNDLE, experiment='shared_detach', seed=CONFIG.seed
)
release_model(detach_model)
print('Saved:', CONFIG.checkpoint_path('shared_detach'))

## 3. Shared localization head + end-to-end auxiliary path

이 셀은 `shared_e2e`만 학습합니다.

In [ ]:
e2e_model, e2e_history, e2e_gradients = train_one_experiment(
    CONFIG, BUNDLE, experiment='shared_e2e', seed=CONFIG.seed
)
release_model(e2e_model)
print('Saved:', CONFIG.checkpoint_path('shared_e2e'))

## 4. Optional ablations

필요할 때 아래 셀을 개별 실행합니다. `random_patch`는 aligned와 동일한 target subset을 사용하고 feature cell만 random화합니다.

In [ ]:
# decay_model, decay_history, decay_gradients = train_one_experiment(
#     CONFIG, BUNDLE, experiment='shared_decay', seed=CONFIG.seed
# )
# release_model(decay_model)

In [ ]:
# random_model, random_history, random_gradients = train_one_experiment(
#     CONFIG, BUNDLE, experiment='random_patch', seed=CONFIG.seed
# )
# release_model(random_model)

## 5. 저장된 결과 평가

학습이 끝난 모델만 `AVAILABLE_EXPERIMENTS`에 남겨 두고 실행합니다. 평가는 항상 labels 없이 main query 경로로 수행합니다.

In [ ]:
AVAILABLE_EXPERIMENTS = ['baseline', 'shared_detach', 'shared_e2e']
history_df = load_histories(CONFIG, AVAILABLE_EXPERIMENTS)
gradient_frames = []
for experiment in AVAILABLE_EXPERIMENTS:
    path = CONFIG.gradients_path(experiment)
    if path.exists() and path.stat().st_size > 0:
        gradient_frames.append(pd.read_csv(path))
gradients_df = pd.concat(gradient_frames, ignore_index=True) if gradient_frames else pd.DataFrame()

display(summarize_histories(CONFIG, history_df))
plot_histories(CONFIG, history_df)
plot_gradient_history(CONFIG, gradients_df)
display(history_df)
display(gradients_df)

In [ ]:
# 저장된 checkpoint를 이용한 main-only validation 재확인
evaluation_rows = [
    evaluate_saved_experiment(CONFIG, BUNDLE, experiment)
    for experiment in AVAILABLE_EXPERIMENTS
]
evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)
evaluation_df.to_csv(CONFIG.output_dir / f'evaluation_{CONFIG.run_mode}.csv', index=False)

In [ ]:
# 원하는 checkpoint 하나의 정성적 결과를 확인합니다.
visualize_main_predictions(CONFIG, BUNDLE, experiment='shared_e2e', count=4, threshold=0.4)